# Adaptive Hyper-Ellipsoidal Interval Type-2 Fuzzy Cluster Centers
## Option 2: Nucleus as the Actual Fuzzy Center
This notebook implements the updated proposed method where the **inner nucleus/hyper-ellipsoid is treated as the actual fuzzy center region**, and the outer hyper-ellipsoid is treated as the **Type-2 uncertainty boundary**.

In [9]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Adaptive Hyper-Ellipsoidal Interval Type-2 Fuzzy Cluster Centers
Option 2: Nucleus as the Actual Fuzzy Center

Updated idea:
- Inner ellipse / hyper-ellipsoid = actual fuzzy center region
- Outer ellipse / hyper-ellipsoid = Type-2 uncertainty boundary
- Nucleus = most reliable center location
- Type-2 region = all plausible center locations
- Outside Type-2 region = unlikely center locations

Assignment rule:
D_j(x) = min_{z in N_j} (x-z)^T Sigma_j^{-1} (x-z)

For a Mahalanobis ellipsoidal nucleus, this reduces to:
- If x lies inside the nucleus, D_j(x) = 0
- If x lies outside the nucleus, D_j(x) = (sqrt(q_j(x)) - sqrt(tau_L_j))^2
where q_j(x) = (x-c_j)^T Sigma_j^{-1} (x-c_j)
"""

import json
import time
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

warnings.filterwarnings("ignore")


# ============================================================
# USER SETTINGS
# ============================================================

DATASET_PATH = "glass.csv"  # Example: "heart (2), iris(3) car (4), glass (6)"
N_CLUSTERS = 6
LABEL_COLUMN = "Type"          # Example: "target"; keep None if no class label heart: target, iris: variety, car:acceptability, glass:Type

FEATURE_COLUMNS = None        # Example: ["age", "bp", "glucose"]; keep None for all numeric columns
MAX_ITER = 100
TOL = 1e-5
RANDOM_STATE = 42

# Inner nucleus = actual fuzzy center region
NUCLEUS_QUANTILE = 0.05

# Outer Type-2 region = uncertainty boundary
TYPE2_QUANTILE = 0.30

# Covariance regularization
EPSILON = 1e-4

SAVE_PREFIX = "Proposed_2"


# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def make_result_folder(dataset_path):
    dataset_name = Path(dataset_path).stem
    result_dir = Path(f"Result_{dataset_name}")
    result_dir.mkdir(parents=True, exist_ok=True)
    return result_dir


def setup_logger(result_dir):
    log_path = result_dir / "run_log.txt"

    logger = logging.getLogger("Adaptive_IT2_NucleusCenter")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    file_handler = logging.FileHandler(log_path, mode="w", encoding="utf-8")
    file_handler.setFormatter(formatter)

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)

    return logger


def load_dataset(dataset_path, label_column=None, feature_columns=None):
    path = Path(dataset_path)

    if not path.exists():
        raise FileNotFoundError(f"Dataset file not found: {dataset_path}")

    if path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    elif path.suffix.lower() in [".csv", ".txt"]:
        df = pd.read_csv(path)
    else:
        raise ValueError("Unsupported file format. Please use CSV, TXT, XLS, or XLSX.")

    df_original = df.copy()

    # Convert categorical columns to numerical values
    for col in df.columns:
        if df[col].dtype == "object" or df[col].dtype.name == "category":
            df[col] = LabelEncoder().fit_transform(df[col].astype(str))

    y_true = None
    if label_column is not None and label_column in df.columns:
        y_true = df[label_column].values
        df = df.drop(columns=[label_column])

    if feature_columns is not None:
        missing_cols = [c for c in feature_columns if c not in df.columns]
        if missing_cols:
            raise ValueError(f"Feature columns not found: {missing_cols}")
        X_df = df[feature_columns].copy()
    else:
        X_df = df.select_dtypes(include=[np.number]).copy()

    if X_df.shape[1] == 0:
        raise ValueError("No numeric feature columns found.")

    X_df = X_df.replace([np.inf, -np.inf], np.nan)
    X_df = X_df.fillna(X_df.median(numeric_only=True))

    feature_names = list(X_df.columns)
    X = X_df.values.astype(float)

    return df_original, X_df, X, y_true, feature_names


def compute_covariance(X_cluster, center, epsilon):
    d = len(center)

    if X_cluster.shape[0] <= 1:
        cov = np.eye(d)
    else:
        centered = X_cluster - center
        cov = (centered.T @ centered) / max(X_cluster.shape[0] - 1, 1)

    return cov + epsilon * np.eye(d)


def mahalanobis_q_matrix(X, centers, covariances):
    """
    q_j(x_i) = (x_i-c_j)^T Sigma_j^{-1} (x_i-c_j)
    """
    N = X.shape[0]
    K = centers.shape[0]
    Q = np.zeros((N, K))

    for j in range(K):
        diff = X - centers[j]
        inv_cov = np.linalg.pinv(covariances[j])
        Q[:, j] = np.sum((diff @ inv_cov) * diff, axis=1)

    return Q


def nucleus_region_distance_matrix(X, centers, covariances, tau_L):
    """
    Option 2 distance:
    D_j(x) = min_{z in N_j} (x-z)^T Sigma_j^{-1} (x-z)

    For ellipsoidal nucleus:
    q = (x-c)^T Sigma^{-1} (x-c)

    If q <= tau_L, x is inside the fuzzy-center nucleus:
        D = 0
    Else:
        D = (sqrt(q) - sqrt(tau_L))^2
    """
    Q = mahalanobis_q_matrix(X, centers, covariances)
    D = np.zeros_like(Q)

    for j in range(centers.shape[0]):
        tau = tau_L[j]
        if not np.isfinite(tau) or tau <= 0:
            D[:, j] = Q[:, j]
        else:
            D[:, j] = np.maximum(0.0, np.sqrt(np.maximum(Q[:, j], 0.0)) - np.sqrt(tau)) ** 2

    return D, Q


def safe_metric(metric_fn, X, labels):
    try:
        if len(np.unique(labels)) < 2:
            return np.nan
        return metric_fn(X, labels)
    except Exception:
        return np.nan


def set_plot_style():
    plt.rcParams.update({
        "font.size": 14,
        "axes.labelsize": 14,
        "axes.labelweight": "bold",
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 10,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    })


def draw_cov_ellipse(ax, mean, cov, tau, edgecolor, linestyle, label=None, linewidth=2.0):
    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.maximum(eigvals, 1e-12)

    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    theta = np.linspace(0, 2 * np.pi, 300)
    circle = np.vstack([np.cos(theta), np.sin(theta)])

    ellipse = eigvecs @ np.diag(np.sqrt(tau * eigvals)) @ circle
    ellipse = ellipse + mean.reshape(2, 1)

    ax.plot(
        ellipse[0, :],
        ellipse[1, :],
        color=edgecolor,
        linestyle=linestyle,
        linewidth=linewidth,
        label=label
    )


# ============================================================
# PROPOSED MODEL
# ============================================================

class AdaptiveHyperEllipsoidalIT2NucleusCenter:
    """
    Proposed:The cluster nucleus is treated as the actual fuzzy center region.
    During reassignment, each point is assigned using its minimum Mahalanobis distance to the nucleus, not only to the crisp centroid.
    """

    def __init__(
        self,
        n_clusters=3,
        max_iter=100,
        tol=1e-5,
        epsilon=1e-4,
        nucleus_quantile=0.40,
        type2_quantile=0.80,
        random_state=42,
        logger=None
    ):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.epsilon = epsilon
        self.nucleus_quantile = nucleus_quantile
        self.type2_quantile = type2_quantile
        self.random_state = random_state
        self.logger = logger

        self.centers_ = None
        self.covariances_ = None
        self.labels_ = None
        self.q_distances_ = None
        self.nucleus_region_distances_ = None
        self.tau_L_ = None
        self.tau_U_ = None
        self.n_iter_ = 0
        self.center_shift_history_ = []
        self.interpretability_ = None

    def _estimate_regions(self, X, labels, centers, covariances):
        Q = mahalanobis_q_matrix(X, centers, covariances)

        tau_L = np.zeros(self.n_clusters)
        tau_U = np.zeros(self.n_clusters)

        for j in range(self.n_clusters):
            idx = np.where(labels == j)[0]
            qj = Q[idx, j]

            if len(qj) == 0:
                tau_L[j] = np.nan
                tau_U[j] = np.nan
            else:
                tau_L[j] = np.quantile(qj, self.nucleus_quantile)
                tau_U[j] = np.quantile(qj, self.type2_quantile)

                if tau_U[j] <= tau_L[j]:
                    tau_U[j] = tau_L[j] + self.epsilon

        return tau_L, tau_U, Q

    def fit(self, X):
        N, d = X.shape
        K = self.n_clusters

        if self.logger:
            self.logger.info("Starting K-means initialization.")

        km = KMeans(n_clusters=K, random_state=self.random_state, n_init=10)
        labels = km.fit_predict(X)
        centers = km.cluster_centers_.copy()

        covariances = np.array([
            compute_covariance(X[labels == j], centers[j], self.epsilon)
            for j in range(K)
        ])

        tau_L, tau_U, Q = self._estimate_regions(X, labels, centers, covariances)
        rng = np.random.default_rng(self.random_state)

        for iteration in range(1, self.max_iter + 1):
            old_centers = centers.copy()
            old_labels = labels.copy()

            for j in range(K):
                Xj = X[labels == j]

                if Xj.shape[0] == 0:
                    idx = rng.integers(0, N)
                    centers[j] = X[idx]
                    Xj = X[[idx]]

                centers[j] = Xj.mean(axis=0)
                covariances[j] = compute_covariance(Xj, centers[j], self.epsilon)

            tau_L, tau_U, Q = self._estimate_regions(X, labels, centers, covariances)

            D_region, Q = nucleus_region_distance_matrix(X, centers, covariances, tau_L)
            labels = np.argmin(D_region, axis=1)

            center_shift = np.linalg.norm(centers - old_centers)
            label_changes = int(np.sum(labels != old_labels))
            self.center_shift_history_.append(center_shift)

            if self.logger:
                self.logger.info(
                    f"Iteration {iteration:03d} | center shift = {center_shift:.8f} | label changes = {label_changes}"
                )

            if center_shift < self.tol and label_changes == 0:
                if self.logger:
                    self.logger.info("Convergence reached.")
                break

        for j in range(K):
            Xj = X[labels == j]
            if Xj.shape[0] > 0:
                centers[j] = Xj.mean(axis=0)
                covariances[j] = compute_covariance(Xj, centers[j], self.epsilon)

        tau_L, tau_U, Q = self._estimate_regions(X, labels, centers, covariances)
        D_region, Q = nucleus_region_distance_matrix(X, centers, covariances, tau_L)

        self.centers_ = centers
        self.covariances_ = covariances
        self.labels_ = labels
        self.q_distances_ = Q
        self.nucleus_region_distances_ = D_region
        self.tau_L_ = tau_L
        self.tau_U_ = tau_U
        self.n_iter_ = iteration

        self._compute_interpretability()
        return self

    def _compute_interpretability(self):
        rows = []

        for j in range(self.n_clusters):
            idx = np.where(self.labels_ == j)[0]
            qj = self.q_distances_[idx, j]

            if len(qj) == 0:
                cluster_count = 0
                nucleus_count = 0
                type2_count = 0
                outside_type2_count = 0
                nd = np.nan
                tc = np.nan
                outside_ratio = np.nan
            else:
                cluster_count = len(qj)
                nucleus_count = int(np.sum(qj <= self.tau_L_[j]))
                type2_count = int(np.sum(qj <= self.tau_U_[j]))
                outside_type2_count = int(np.sum(qj > self.tau_U_[j]))

                nd = nucleus_count / cluster_count
                tc = type2_count / cluster_count
                outside_ratio = outside_type2_count / cluster_count

            rows.append({
                "Cluster": j,
                "Cluster_Size": cluster_count,
                "Tau_L_Fuzzy_Center_Nucleus": self.tau_L_[j],
                "Tau_U_Type2_Uncertainty_Boundary": self.tau_U_[j],
                "Fuzzy_Center_Nucleus_Count": nucleus_count,
                "Type2_Region_Count": type2_count,
                "Outside_Type2_Count": outside_type2_count,
                "Nucleus_Density_ND": nd,
                "Type2_Coverage_TC": tc,
                "Outside_Type2_Ratio": outside_ratio
            })

        self.interpretability_ = pd.DataFrame(rows)


# ============================================================
# RESULT SAVING
# ============================================================

def save_results(result_dir, model, X_scaled, X_df, y_true, feature_names, runtime_seconds):
    labels = model.labels_
    K = model.n_clusters
    N, d = X_scaled.shape

    theoretical_complexity = "O(T N K d^2)"

    label_df = X_df.copy()
    label_df["Cluster_Label"] = labels
    label_df.to_csv(result_dir / f"{SAVE_PREFIX}_cluster_labels.csv", index=False)

    centers_df = pd.DataFrame(model.centers_, columns=feature_names)
    centers_df.insert(0, "Cluster", np.arange(K))
    centers_df.to_csv(result_dir / f"{SAVE_PREFIX}_cluster_centers_standardized.csv", index=False)

    np.savez(result_dir / f"{SAVE_PREFIX}_covariance_matrices.npz", covariances=model.covariances_)

    q_df = pd.DataFrame(
        model.q_distances_,
        columns=[f"Mahalanobis_Q_to_Cluster_{j}" for j in range(K)]
    )
    q_df["Assigned_Cluster"] = labels
    q_df.to_csv(result_dir / f"{SAVE_PREFIX}_mahalanobis_q_distances.csv", index=False)

    region_df = pd.DataFrame(
        model.nucleus_region_distances_,
        columns=[f"Distance_to_Fuzzy_Center_Nucleus_{j}" for j in range(K)]
    )
    region_df["Assigned_Cluster"] = labels
    region_df.to_csv(result_dir / f"{SAVE_PREFIX}_nucleus_region_distances.csv", index=False)

    model.interpretability_.to_csv(
        result_dir / f"{SAVE_PREFIX}_interpretability_measures.csv",
        index=False
    )

    history_df = pd.DataFrame({
        "Iteration": np.arange(1, len(model.center_shift_history_) + 1),
        "Center_Shift": model.center_shift_history_
    })
    history_df.to_csv(result_dir / f"{SAVE_PREFIX}_convergence_history.csv", index=False)

    metrics = {
        "N_Samples": int(N),
        "N_Features": int(d),
        "N_Clusters": int(K),
        "Iterations": int(model.n_iter_),
        "Runtime_Seconds": float(runtime_seconds),
        "Silhouette_Score": float(safe_metric(silhouette_score, X_scaled, labels)),
        "Davies_Bouldin_Index": float(safe_metric(davies_bouldin_score, X_scaled, labels)),
        "Calinski_Harabasz_Index": float(safe_metric(calinski_harabasz_score, X_scaled, labels)),
        "Theoretical_Time_Complexity": theoretical_complexity,
        "Fuzzy_Center_Definition": "Inner nucleus/hyper-ellipsoid is treated as the actual fuzzy center region.",
        "Type2_Region_Definition": "Outer hyper-ellipsoid is treated as Type-2 uncertainty boundary."
    }

    if y_true is not None:
        metrics["Adjusted_Rand_Index"] = float(adjusted_rand_score(y_true, labels))
        metrics["Normalized_Mutual_Info"] = float(normalized_mutual_info_score(y_true, labels))

    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(result_dir / f"{SAVE_PREFIX}_validation_metrics.csv", index=False)

    with open(result_dir / f"{SAVE_PREFIX}_validation_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=4)

    complexity_report = {
        "Theoretical_Time_Complexity": theoretical_complexity,
        "Reason": "Full covariance matrices and Mahalanobis-type region distance are used.",
        "T_iterations": int(model.n_iter_),
        "N_samples": int(N),
        "K_clusters": int(K),
        "d_features": int(d),
        "Approximate_Operation_Form": "T * N * K * d^2",
        "Approximate_Operation_Count": int(model.n_iter_ * N * K * (d ** 2)),
        "Runtime_Seconds": float(runtime_seconds)
    }

    with open(result_dir / f"{SAVE_PREFIX}_time_complexity.json", "w", encoding="utf-8") as f:
        json.dump(complexity_report, f, indent=4)

    with open(result_dir / f"{SAVE_PREFIX}_time_complexity.txt", "w", encoding="utf-8") as f:
        f.write("Time Complexity Report\n")
        f.write("======================\n\n")
        f.write(f"Theoretical time complexity: {theoretical_complexity}\n")
        f.write("Reason: Full covariance matrices and Mahalanobis-type region distance are used.\n\n")
        f.write(f"T iterations: {model.n_iter_}\n")
        f.write(f"N samples: {N}\n")
        f.write(f"K clusters: {K}\n")
        f.write(f"d features: {d}\n")
        f.write(f"Approximate operation count: {model.n_iter_ * N * K * (d ** 2)}\n")
        f.write(f"Runtime seconds: {runtime_seconds:.6f}\n")

    return metrics_df


# ============================================================
# PLOTTING FUNCTIONS
# ============================================================

def plot_clusters_with_fuzzy_centers(result_dir, model, X_scaled):
    set_plot_style()

    N, d = X_scaled.shape
    labels = model.labels_
    K = model.n_clusters

    if d > 2:
        pca = PCA(n_components=2, random_state=RANDOM_STATE)
        X_plot = pca.fit_transform(X_scaled)
        centers_plot = pca.transform(model.centers_)
        xlabel = "PC1"
        ylabel = "PC2"
    else:
        X_plot = X_scaled[:, :2]
        centers_plot = model.centers_[:, :2]
        xlabel = "Feature 1"
        ylabel = "Feature 2"

    light_colors = [
        "#A6CEE3", "#B2DF8A", "#FDBF6F", "#CAB2D6", "#FFFF99",
        "#FB9A99", "#CCEBC5", "#DECBE4", "#FED9A6", "#B3CDE3"
    ]

    edge_colors = [
        "#1F78B4", "#33A02C", "#FF7F00", "#6A3D9A", "#B15928",
        "#E31A1C", "#4DAF4A", "#984EA3", "#A65628", "#377EB8"
    ]

    fig, ax = plt.subplots(figsize=(8, 6))

    for j in range(K):
        idx = labels == j
        color = light_colors[j % len(light_colors)]
        edge = edge_colors[j % len(edge_colors)]

        ax.scatter(
            X_plot[idx, 0],
            X_plot[idx, 1],
            s=35,
            color=color,
            edgecolor="white",
            linewidth=0.4,
            alpha=0.85,
            label=f"Cluster {j}"
        )

        if np.sum(idx) > 2:
            cov_2d = np.cov(X_plot[idx].T) + EPSILON * np.eye(2)
        else:
            cov_2d = np.eye(2) * 0.05

        center_2d = centers_plot[j]
        tau_l = model.tau_L_[j]
        tau_u = model.tau_U_[j]

        if np.isfinite(tau_l) and tau_l > 0:
            draw_cov_ellipse(
                ax,
                center_2d,
                cov_2d,
                tau_l,
                edgecolor=edge,
                linestyle="-",
                label=f"Fuzzy Center Nucleus {j}",
                linewidth=2.4
            )

        if np.isfinite(tau_u) and tau_u > 0:
            draw_cov_ellipse(
                ax,
                center_2d,
                cov_2d,
                tau_u,
                edgecolor=edge,
                linestyle="--",
                label=f"Type-2 Boundary {j}",
                linewidth=2.0
            )

        ax.scatter(
            center_2d[0],
            center_2d[1],
            marker="X",
            s=180,
            color=edge,
            edgecolor="black",
            linewidth=1.0,
            zorder=5
        )

        ax.text(
            center_2d[0],
            center_2d[1],
            f" C{j}",
            fontsize=14,
            fontweight="bold",
            color="black"
        )

    ax.set_xlabel(xlabel, fontsize=14, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=14, fontweight="bold")
    ax.set_title(
        "Nucleus as Fuzzy Center",
        fontsize=14,
        fontweight="bold"
    )

    ax.grid(True, linestyle="--", alpha=0.30)

    handles, labels_legend = ax.get_legend_handles_labels()
    unique = dict(zip(labels_legend, handles))

    ax.legend(
        unique.values(),
        unique.keys(),
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=True,
        fontsize=9
    )

    plt.tight_layout()
    plt.savefig(
        result_dir / f"{SAVE_PREFIX}_fuzzy_center_nucleus_plot.pdf",
        format="pdf",
        bbox_inches="tight"
    )
    plt.close()


def plot_interpretability_bars(result_dir, model):
    set_plot_style()

    df = model.interpretability_.copy()
    x = np.arange(len(df))
    width = 0.25

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.bar(
        x - width,
        df["Nucleus_Density_ND"],
        width,
        label="Nucleus Density",
        color="#B3CDE3",
        edgecolor="black",
        linewidth=0.5
    )

    ax.bar(
        x,
        df["Type2_Coverage_TC"],
        width,
        label="Type-2 Coverage",
        color="#CCEBC5",
        edgecolor="black",
        linewidth=0.5
    )

    ax.bar(
        x + width,
        df["Outside_Type2_Ratio"],
        width,
        label="Outside Type-2 Ratio",
        color="#FED9A6",
        edgecolor="black",
        linewidth=0.5
    )

    ax.set_xlabel("Cluster", fontsize=14, fontweight="bold")
    ax.set_ylabel("Score", fontsize=14, fontweight="bold")
    ax.set_title("Interpretability Measures", fontsize=14, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([f"C{int(c)}" for c in df["Cluster"]], fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.30)

    plt.tight_layout()
    plt.savefig(
        result_dir / f"{SAVE_PREFIX}_interpretability_bar_plot.pdf",
        format="pdf",
        bbox_inches="tight"
    )
    plt.close()


def plot_cluster_sizes(result_dir, model):
    set_plot_style()

    df = model.interpretability_.copy()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(
        df["Cluster"].astype(str),
        df["Cluster_Size"],
        color="#FDBF6F",
        edgecolor="black",
        linewidth=0.5
    )

    ax.set_xlabel("Cluster", fontsize=14, fontweight="bold")
    ax.set_ylabel("Number of Points", fontsize=14, fontweight="bold")
    ax.set_title("Cluster Size Distribution", fontsize=14, fontweight="bold")
    ax.grid(axis="y", linestyle="--", alpha=0.30)

    plt.tight_layout()
    plt.savefig(
        result_dir / f"{SAVE_PREFIX}_cluster_size_plot.pdf",
        format="pdf",
        bbox_inches="tight"
    )
    plt.close()


def plot_convergence(result_dir, model):
    set_plot_style()

    history = np.array(model.center_shift_history_)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(
        np.arange(1, len(history) + 1),
        history,
        marker="o",
        linewidth=2.0,
        color="#A6CEE3",
        markeredgecolor="black"
    )

    ax.set_xlabel("Iteration", fontsize=14, fontweight="bold")
    ax.set_ylabel("Center Shift", fontsize=14, fontweight="bold")
    ax.set_title("Convergence Curve", fontsize=14, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.30)

    plt.tight_layout()
    plt.savefig(
        result_dir / f"{SAVE_PREFIX}_convergence_curve.pdf",
        format="pdf",
        bbox_inches="tight"
    )
    plt.close()


def plot_validation_metrics(result_dir, metrics_df):
    set_plot_style()

    metric_names = [
        "Silhouette_Score",
        "Davies_Bouldin_Index",
        "Calinski_Harabasz_Index"
    ]

    available_metrics = [m for m in metric_names if m in metrics_df.columns]
    values = [metrics_df[m].iloc[0] for m in available_metrics]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(
        available_metrics,
        values,
        color=["#B3CDE3", "#CCEBC5", "#DECBE4"],
        edgecolor="black",
        linewidth=0.5
    )

    ax.set_xlabel("Validation Index", fontsize=14, fontweight="bold")
    ax.set_ylabel("Value", fontsize=14, fontweight="bold")
    ax.set_title("Clustering Validation Metrics", fontsize=14, fontweight="bold")
    ax.tick_params(axis="x", labelrotation=20)
    ax.grid(axis="y", linestyle="--", alpha=0.30)

    plt.tight_layout()
    plt.savefig(
        result_dir / f"{SAVE_PREFIX}_validation_metrics_plot.pdf",
        format="pdf",
        bbox_inches="tight"
    )
    plt.close()


# ============================================================
# MAIN FUNCTION
# ============================================================

def main():
    result_dir = make_result_folder(DATASET_PATH)
    logger = setup_logger(result_dir)

    logger.info("==================================================")
    logger.info("Option-2 Adaptive Hyper-Ellipsoidal IT2 Fuzzy Center")
    logger.info("Nucleus is treated as the actual fuzzy center region.")
    logger.info("==================================================")
    logger.info(f"Dataset path: {DATASET_PATH}")
    logger.info(f"Result folder: {result_dir}")
    logger.info(f"DATASET_PATH = {DATASET_PATH}")
    logger.info(f"N_CLUSTERS = {N_CLUSTERS}")
    logger.info(f"LABEL_COLUMN = {LABEL_COLUMN}")

    start_time = time.time()

    df_original, X_df, X, y_true, feature_names = load_dataset(
        DATASET_PATH,
        label_column=LABEL_COLUMN,
        feature_columns=FEATURE_COLUMNS
    )

    logger.info(f"Original dataset shape: {df_original.shape}")
    logger.info(f"Numeric feature matrix shape: {X.shape}")
    logger.info(f"Number of clusters: {N_CLUSTERS}")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    preprocessing_info = {
        "Dataset_Path": DATASET_PATH,
        "Original_Rows": int(df_original.shape[0]),
        "Original_Columns": int(df_original.shape[1]),
        "Used_Features": feature_names,
        "Number_of_Used_Features": int(len(feature_names)),
        "Label_Column": LABEL_COLUMN,
        "Missing_Value_Strategy": "Median imputation for numeric columns",
        "Categorical_Handling": "LabelEncoder for categorical columns",
        "Scaling": "StandardScaler"
    }

    with open(result_dir / f"{SAVE_PREFIX}_preprocessing_info.json", "w", encoding="utf-8") as f:
        json.dump(preprocessing_info, f, indent=4)

    X_df.to_csv(result_dir / f"{SAVE_PREFIX}_used_numeric_features.csv", index=False)

    model = AdaptiveHyperEllipsoidalIT2NucleusCenter(
        n_clusters=N_CLUSTERS,
        max_iter=MAX_ITER,
        tol=TOL,
        epsilon=EPSILON,
        nucleus_quantile=NUCLEUS_QUANTILE,
        type2_quantile=TYPE2_QUANTILE,
        random_state=RANDOM_STATE,
        logger=logger
    )

    model.fit(X_scaled)

    runtime_seconds = time.time() - start_time
    logger.info(f"Finished clustering in {runtime_seconds:.6f} seconds.")
    logger.info("Theoretical time complexity: O(T N K d^2)")
    logger.info("Assignment rule: D_j(x)=min_{z in N_j}(x-z)^T Sigma_j^{-1}(x-z)")

    metrics_df = save_results(
        result_dir=result_dir,
        model=model,
        X_scaled=X_scaled,
        X_df=X_df,
        y_true=y_true,
        feature_names=feature_names,
        runtime_seconds=runtime_seconds
    )

    logger.info("Saved numerical results.")

    plot_clusters_with_fuzzy_centers(result_dir, model, X_scaled)
    plot_interpretability_bars(result_dir, model)
    plot_cluster_sizes(result_dir, model)
    plot_convergence(result_dir, model)
    plot_validation_metrics(result_dir, metrics_df)

    logger.info("Saved all plots as PDF files.")

    summary = {
        "Status": "Completed",
        "Dataset": DATASET_PATH,
        "Result_Folder": str(result_dir),
        "N_Clusters": N_CLUSTERS,
        "Iterations": model.n_iter_,
        "Runtime_Seconds": runtime_seconds,
        "Main_Update": "Option 2: Nucleus is treated as the actual fuzzy center region.",
        "Assignment_Rule": "D_j(x)=min_{z in N_j}(x-z)^T Sigma_j^{-1}(x-z)",
        "Output_Files": [
            f"{SAVE_PREFIX}_cluster_labels.csv",
            f"{SAVE_PREFIX}_cluster_centers_standardized.csv",
            f"{SAVE_PREFIX}_covariance_matrices.npz",
            f"{SAVE_PREFIX}_mahalanobis_q_distances.csv",
            f"{SAVE_PREFIX}_nucleus_region_distances.csv",
            f"{SAVE_PREFIX}_interpretability_measures.csv",
            f"{SAVE_PREFIX}_validation_metrics.csv",
            f"{SAVE_PREFIX}_validation_metrics.json",
            f"{SAVE_PREFIX}_time_complexity.txt",
            f"{SAVE_PREFIX}_time_complexity.json",
            f"{SAVE_PREFIX}_fuzzy_center_nucleus_plot.pdf",
            f"{SAVE_PREFIX}_interpretability_bar_plot.pdf",
            f"{SAVE_PREFIX}_cluster_size_plot.pdf",
            f"{SAVE_PREFIX}_convergence_curve.pdf",
            f"{SAVE_PREFIX}_validation_metrics_plot.pdf",
            "run_log.txt"
        ]
    }

    with open(result_dir / f"{SAVE_PREFIX}_final_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=4)

    logger.info("Final summary saved.")
    logger.info("All results stored successfully.")
    logger.info("==================================================")


main()


2026-06-23 12:21:22 | INFO | ==================================================
2026-06-23 12:21:22 | INFO | Option-2 Adaptive Hyper-Ellipsoidal IT2 Fuzzy Center
2026-06-23 12:21:22 | INFO | Nucleus is treated as the actual fuzzy center region.
2026-06-23 12:21:22 | INFO | ==================================================
2026-06-23 12:21:22 | INFO | Dataset path: glass.csv
2026-06-23 12:21:22 | INFO | Result folder: Result_glass
2026-06-23 12:21:22 | INFO | DATASET_PATH = glass.csv
2026-06-23 12:21:22 | INFO | N_CLUSTERS = 6
2026-06-23 12:21:22 | INFO | LABEL_COLUMN = Type
2026-06-23 12:21:22 | INFO | Original dataset shape: (214, 10)
2026-06-23 12:21:22 | INFO | Numeric feature matrix shape: (214, 9)
2026-06-23 12:21:22 | INFO | Number of clusters: 6
2026-06-23 12:21:22 | INFO | Starting K-means initialization.
2026-06-23 12:21:22 | INFO | Iteration 001 | center shift = 0.00000000 | label changes = 26
2026-06-23 12:21:22 | INFO | Iteration 002 | center shift = 0.64215761 | label cha